In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openpyxl import Workbook
from openbabel import pybel
import shutil
import time
import pandas as pd
from rdkit import Chem

In [ ]:
os.environ['PATH'] = '/home/fwtop/apps/openmpi/bin:' + os.environ.get('PATH', '')

os.environ['LD_LIBRARY_PATH'] = '/home/fwtop/apps/openmpi/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

os.environ['OMPI_ALLOW_RUN_AS_ROOT'] = '1'
os.environ['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'

In [ ]:

MAX_TASKS = 2

In [ ]:

nproc = 10
mem = 3000  

In [ ]:



os.makedirs('energy/neutral/success', exist_ok=True)
os.makedirs('energy/neutral/failure', exist_ok=True)


os.makedirs('energy/oxidation/success', exist_ok=True)
os.makedirs('energy/oxidation/failure', exist_ok=True)


os.makedirs('energy/reduction/success', exist_ok=True)
os.makedirs('energy/reduction/failure', exist_ok=True)

In [ ]:

opt_freq_path = "opt+freq/success"
neutral_energy_path = "energy/neutral"
oxidation_energy_path = "energy/oxidation"
reduction_energy_path = "energy/reduction"

In [ ]:

def get_charge_and_multiplicity(opt_freq_path, df):

    
    result_dict = {}

    
    for filename in os.listdir(opt_freq_path):
        if filename.endswith('.inp'):
            
            name = os.path.splitext(filename)[0]

            
            row = df[df['Name'] == name]
            if row.empty:
                continue  

            
            smiles = row['SMILES'].values[0]

            
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                continue  

            
            mol = Chem.AddHs(mol)

            
            neutral_charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())

            
            neutral_multiplicity = sum(atom.GetNumRadicalElectrons() for atom in mol.GetAtoms()) + 1

            
            oxidized_charge = neutral_charge + 1  
            oxidized_multiplicity = neutral_multiplicity - 1 if neutral_multiplicity > 1 else neutral_multiplicity + 1

            
            reduced_charge = neutral_charge - 1  
            reduced_multiplicity = neutral_multiplicity - 1 if neutral_multiplicity > 1 else neutral_multiplicity + 1

            
            charge_multiplicity_dict = {
                'neutral': f"{neutral_charge} {neutral_multiplicity}",
                'oxidized': f"{oxidized_charge} {oxidized_multiplicity}",
                'reduced': f"{reduced_charge} {reduced_multiplicity}"
            }

            
            result_dict[name] = charge_multiplicity_dict

    return result_dict

In [ ]:


def extract_opt_coordinates(xyz_file):
    with open(xyz_file, 'r') as file:
        lines = file.readlines()
    
    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*$')
    
    
    start_index = None
    for index, line in enumerate(lines):
        if atom_line_pattern.match(line):
            start_index = index
            break
    
    
    final_index = None
    for index, line in enumerate(lines[start_index + 1 : ]):
        if atom_line_pattern.match(line) == False:
            final_index = index
            break
    
    
    coordinates_lines = lines[start_index : final_index]
    
    
    coordinates_str = ''.join(coordinates_lines)
    
    return coordinates_str

In [ ]:

def extract_inp_charge_and_multiplicity_line(inp_file):
    with open(inp_file, 'r') as file:
        lines = file.readlines()
    
    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*$')
    
    
    start_index = None
    for index, line in enumerate(lines):
        if atom_line_pattern.match(line):
            start_index = index
            break
    
    
    charge_and_multiplicity_line = lines[start_index - 1]        
    
    return charge_and_multiplicity_line

In [ ]:

def create_energy_inp(coordinates, charge_and_multiplicity, energy_inp_file, mem=mem, nproc=nproc):
    
    
    
    with open(energy_inp_file, 'w') as f:
        f.write(f"! wB97M-V ma-def2-TZVP autoaux RIJCOSX strongSCF noautostart miniprint\n")
        f.write(f"%maxcore     {mem}\n")
        f.write(f"%pal nprocs  {nproc} end\n")
        f.write(f"* xyz {charge_and_multiplicity}\n") 
        f.write(coordinates)
        f.write(f"*")

In [ ]:
def run_orca_gas_energy(energy_inp_file, energy_success_dir, energy_failure_dir):
    
    
    inp_dir = os.path.dirname(energy_inp_file) if os.path.dirname(energy_inp_file) else '.'
    
    
    base_name = os.path.splitext(os.path.basename(energy_inp_file))[0]
    
    
    output_file = os.path.join(inp_dir, base_name + '.out')
    opt_file = os.path.join(inp_dir, base_name + '.opt')
    trj_file = os.path.join(inp_dir, base_name + '_trj' + '.xyz')
    xyz_file = os.path.join(inp_dir, base_name + '.xyz')
    property_file = os.path.join(inp_dir, base_name + '.property' + '.txt')
    gbw_file = os.path.join(inp_dir, base_name + '.gbw')
    engrad_file = os.path.join(inp_dir, base_name + '.engrad')
    densitiesinfo_file = os.path.join(inp_dir, base_name + '.densitiesinfo')
    densities_file = os.path.join(inp_dir, base_name + '.densities')
    hess_file = os.path.join(inp_dir, base_name + '.hess')
    bibtex_file = os.path.join(inp_dir, base_name + '.bibtex')

    try:
        
        with open(output_file, 'w') as out:
            subprocess.run(['/home/public/orca_6_0_1_linux_x86-64_shared_openmpi416_avx2/orca', energy_inp_file], stdout=out, check=True)

        
        with open(output_file, 'r') as f:
            content = f.read()
            if 'ORCA TERMINATED NORMALLY' in content:
                
                files_to_move = [
                    energy_inp_file, output_file, opt_file, trj_file, xyz_file,
                    property_file, gbw_file, engrad_file, densitiesinfo_file,
                    densities_file, hess_file, bibtex_file
                ]
                for f in files_to_move:
                    if os.path.exists(f):
                        os.rename(f, os.path.join(energy_success_dir, os.path.basename(f)))
                
                new_output_path = os.path.join(energy_success_dir, os.path.basename(output_file))
                return new_output_path, True
            else:
                
                files_to_move = [
                    energy_inp_file, output_file, opt_file, trj_file, xyz_file,
                    property_file, gbw_file, engrad_file, densitiesinfo_file,
                    densities_file, hess_file, bibtex_file
                ]
                for f in files_to_move:
                    if os.path.exists(f):
                        os.rename(f, os.path.join(energy_failure_dir, os.path.basename(f)))
                return None, False

    except subprocess.CalledProcessError as e:
        
        if os.path.exists(energy_inp_file):
            os.rename(energy_inp_file, os.path.join(energy_failure_dir, os.path.basename(energy_inp_file)))
        if os.path.exists(output_file):
            os.rename(output_file, os.path.join(energy_failure_dir, os.path.basename(output_file)))
        print(f"Error running ORCA for file {energy_inp_file}: {str(e)}")
        print(traceback.format_exc())
        return None, False

In [ ]:

def main():
    df = pd.read_excel("HTQC_ox_red.xlsx")
    result_dict = get_charge_and_multiplicity(opt_freq_path, df)
    
    
    for out_file in os.listdir(opt_freq_path):
        if out_file.endswith(".out"):
            
            base_name = os.path.splitext(out_file)[0]
    
            
            opt_xyz_file = os.path.join(opt_freq_path, base_name + ".xyz")
            opt_inp_file = os.path.join(opt_freq_path, base_name + ".inp")
            
            
            if base_name in result_dict:
                
                state_dict = result_dict[base_name]
                
                
                source_file = os.path.join(opt_freq_path, out_file)
                
                
                for state, charge_and_multiplicity in state_dict.items():
                    
                    if state == 'neutral':
                        energy_inp_file = os.path.join(neutral_energy_path, out_file.replace(".out", ".inp"))
                        energy_path_state = neutral_energy_path
                    elif state == 'oxidized':
                        energy_inp_file = os.path.join(oxidation_energy_path, out_file.replace(".out", ".inp"))
                        energy_path_state = oxidation_energy_path
                    elif state == 'reduced':
                        energy_inp_file = os.path.join(reduction_energy_path, out_file.replace(".out", ".inp"))
                        energy_path_state = reduction_energy_path
                    else:
                        continue  
    
                    coordinates = extract_opt_coordinates(opt_xyz_file) 
                    
                    create_energy_inp(coordinates, charge_and_multiplicity, energy_inp_file, mem=mem, nproc=nproc)

    
    
    energy_paths = [neutral_energy_path, oxidation_energy_path, reduction_energy_path]
    
    
    def create_success_failure_dirs(energy_path):
        success_dir = os.path.join(energy_path, "success")
        failure_dir = os.path.join(energy_path, "failure")
        os.makedirs(success_dir, exist_ok=True)
        os.makedirs(failure_dir, exist_ok=True)
        return success_dir, failure_dir
    
    
    def process_energy_path(energy_path):
        
        energy_success_dir, energy_failure_dir = create_success_failure_dirs(energy_path)
    
        
        with ThreadPoolExecutor(max_workers=MAX_TASKS) as executor:
            
            futures = [
                executor.submit(run_orca_gas_energy, os.path.join(energy_path, inp_file), energy_success_dir, energy_failure_dir)
                for inp_file in os.listdir(energy_path) if inp_file.endswith(".inp")
            ]
            
            for future in as_completed(futures):
                pass  
    
    
    with ThreadPoolExecutor(max_workers=1) as executor:
        futures = [executor.submit(process_energy_path, energy_path) for energy_path in energy_paths]
        for future in as_completed(futures):
            pass  

In [ ]:
if __name__ == "__main__":
    main()